# Persian Comment Classification

**Selected Machine Learning Exercise**

A binary text-classification exercise that implements a Naive Bayes-style classifier for Persian comments.

In [78]:
import pandas as pd
import numpy as np

In [ ]:
train = pd.read_csv("data/train.csv")

In [80]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(train, test_size=0.2, random_state=42, stratify=train["price_value"])

In [81]:
train.head()

,comment,price_value
0,قیمت مناسب وکیفیت خوب پیشنهادمیکنم حتما خرید کنید,1
1,به اندازه یک میلیمتر دورتادور گوشی خالی میماند...,0
2,از همه نظر عالی و یک خرید خوب در قیمت حدود۴۰ ...,1
3,فقط یک بار هر یک ربع ساعت 1 درصد شارژ کرد بعدش...,0
4,قیمت این کالا خیلی تغییر میکنه . من خریدم چندر...,1


## calculating prior probability

In [82]:
prior_probability = train_df["price_value"].value_counts(normalize=True).to_dict()

print(prior_probability)

{0: 0.52003125, 1: 0.47996875}


## preprocess function

### Hazm dependency

The `hazm` package is listed in `requirements.txt`. Install the project dependencies before running the notebook.

In [84]:
from hazm import Normalizer, Stemmer, word_tokenize, stopwords_list
import re

In [93]:
import re
from hazm import Normalizer, word_tokenize, stopwords_list

normalizer = Normalizer()

stop_words = set(stopwords_list())
stop_words.discard("قیمت")

normalizer = Normalizer()

stop_words = set(stopwords_list())
stop_words.discard("قیمت")

# فقط یک بار ساخته میشه
token_pattern = re.compile(r"[\u0600-\u06FFA-Za-z0-9]+")


def preprocessing(texts):
    filtered = []

    # هم یک string قبول می‌کنه هم Series/List
    if isinstance(texts, str):
        texts = [texts]

    for text in texts:

        # نرمال‌سازی فارسی
        text = normalizer.normalize(str(text))

        # tokenize + حذف punctuation در یک مرحله
        tokens = token_pattern.findall(text)

        for token in tokens:

            # عددها باقی بمانند
            if token.isdigit():
                filtered.append(token)
                continue

            # قیمت، قیمتش، قیمتشان، قیمت‌ها و ... همگی → قیمت
            if token.startswith("قیمت"):
                filtered.append("قیمت")
                continue

            # stop words
            if token in stop_words:
                continue

            filtered.append(token)

    return filtered

In [94]:
texts = [
    "قیمت این محصول خوبه",
    "قیمتش خیلی مناسبه"]

preprocessing(texts)

['قیمت', 'محصول', 'خوبه', 'قیمت', 'مناسبه']

## how many words are there in each class

In [95]:
from collections import Counter

def token_counter(texts):
    return dict(Counter(preprocessing(texts)))

In [96]:
texts = [
    "قیمت این محصول خوبه",
    "قیمتش خیلی مناسبه"]
token_counter(texts)

{'قیمت': 2, 'محصول': 1, 'خوبه': 1, 'مناسبه': 1}

In [97]:
neg_class = train_df.loc[train_df["price_value"] == 0, "comment"]

pos_class = train_df.loc[train_df["price_value"] == 1, "comment"]

In [98]:
words_in_neg_class = token_counter(neg_class)
words_in_pos_class = token_counter(pos_class)

In [ ]:
# neg_class = train_df[train_df["price_value"] == 0]["comment"]
# pos_class = train_df[train_df["price_value"] == 1]["comment"]

In [ ]:
# words_in_neg_class = token_counter(neg_class)
# words_in_pos_class = token_counter(pos_class)

In [99]:
def compute_probability(text, cls):

    tokens = preprocessing(text)

    if cls == 1:
        class_count = words_in_pos_class
    else:
        class_count = words_in_neg_class

    # تمام کلمات یکتای هر دو کلاس
    vocabulary = set(words_in_pos_class.keys()) | set(words_in_neg_class.keys())

    vocab_size = len(vocabulary)

    # مجموع تعداد تمام توکن‌های این کلاس
    total_tokens = sum(class_count.values())

    # از prior کلاس شروع می‌کنیم
    probability = prior_probability[cls]

    for token in tokens:

        token_count = class_count.get(token, 0)

        token_probability = (
            token_count + 1
        ) / (
            total_tokens + vocab_size
        )

        probability *= token_probability

    return probability

In [100]:
def predict(text):
    p_positive = compute_probability(text, 1)
    p_negative = compute_probability(text, 0)

    if p_positive > p_negative:
        return 1
    else:
        return 0

In [101]:
from sklearn.metrics import accuracy_score

val_predictions = val_df["comment"].apply(predict)

val_accuracy = accuracy_score(val_df["price_value"], val_predictions)

print("Validation accuracy:", val_accuracy)

Validation accuracy: 0.83375


## training on the whole trainset

In [102]:
total = len(train)

prior_probability = {
    0: (train["price_value"] == 0).sum() / total,
    1: (train["price_value"] == 1).sum() / total
}

In [ ]:
words_in_neg_class = token_counter(train[train["price_value"] == 0]["comment"])

words_in_pos_class = token_counter(train[train["price_value"] == 1]["comment"])

In [104]:
train_predictions = train["comment"].apply(predict)

train_accuracy = accuracy_score(train["price_value"], train_predictions)

print("Train accuracy:", train_accuracy)

Train accuracy: 0.884475


## testing on the test set

In [ ]:
test = pd.read_csv("data/test.csv")

In [106]:
test_predictions = test["comment"].apply(predict)

In [107]:
submission = pd.DataFrame({"price_value": test_predictions})

submission.head()

,price_value
0,1
1,1
2,0
3,0
4,1


In [ ]:
from pathlib import Path

Path("outputs").mkdir(exist_ok=True)
submission.to_csv("outputs/submission.csv", index=False)

print("Saved outputs/submission.csv")